# Task 4: Open-Set Recognition (OSR) on CIFAR-10 vs CIFAR-100

This notebook implements the complete benchmark suite for **Open-Set Recognition (OSR)**:
- **Known Classes:** Full 10 CIFAR-10 classes (90/10 stratified split on train set via seed `6304`; evaluation on CIFAR-10 test set).
- **Unknown Classes:** Fixed 16 CIFAR-100 fine classes (800 Near Unknowns, 800 Far Unknowns).
- **Models:**
  1. **Vanilla ResNet-18** (CIFAR-adapted $3\times3$ conv1, no maxpool, cross-entropy, standard crop/flip).
  2. **GCSC** (Vanilla + `RandAugment(num_ops=2, magnitude=9)`).
  3. **PROSER** (Initialized from Vanilla, 5 dummy classifiers, manifold mixup after `layer2` with $\beta=1.0, \gamma=0.1$).
- **Post-hoc Novelty Scores:** $u_{\text{MSP}}$, $u_{\text{MLS}}$, $u_{\text{Energy}}$, $u_{\text{Mahalanobis}}$, and PROSER placeholder score.
- **Diagnostics & Reporting:** AUROC (Near, Far, All), validation-calibrated FPR@95TPR, multi-panel ROC curves, and failure inspection.

## 1. Environment Setup & Directory Scaffold

In [ ]:
import os
import sys

TASK4_DIRS = [
    "task4/models",
    "task4/methods",
    "task4/evaluation",
    "task4/results/checkpoints",
    "task4/results/plots"
]
for d in TASK4_DIRS:
    os.makedirs(d, exist_ok=True)

for p in ["task4/__init__.py", "task4/models/__init__.py", "task4/methods/__init__.py", "task4/evaluation/__init__.py"]:
    with open(p, "a") as f:
        pass

if "." not in sys.path:
    sys.path.insert(0, ".")

print("Task 4 directory structure initialized.")

Task 4 directory structure initialized.


## 2. Dataset Protocol & Model Architectures (`%%writefile`)

In [ ]:
%%writefile task4/dataset_protocol.py
import random
import numpy as np
import torch
from torch.utils.data import Dataset, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedShuffleSplit

SEED = 6304

NEAR_UNKNOWN_CLASSES = [
    "bus", "pickup_truck", "motorcycle", "tractor",
    "wolf", "fox", "leopard", "camel"
]
FAR_UNKNOWN_CLASSES = [
    "bottle", "bowl", "chair", "clock",
    "keyboard", "mushroom", "sunflower", "wardrobe"
]

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

def get_cifar_transforms(use_randaugment=False):
    train_ops = [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip()
    ]
    if use_randaugment:
        train_ops.append(transforms.RandAugment(num_ops=2, magnitude=9))

    train_ops.extend([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
    ])

    test_ops = [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
    ]
    return transforms.Compose(train_ops), transforms.Compose(test_ops)

class IndexedDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img, label, idx

def get_cifar10_splits(data_root="./data", use_randaugment=False):
    train_tf, eval_tf = get_cifar_transforms(use_randaugment=use_randaugment)
    full_train = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tf)
    full_train_eval = datasets.CIFAR10(root=data_root, train=True, download=True, transform=eval_tf)
    test_set = datasets.CIFAR10(root=data_root, train=False, download=True, transform=eval_tf)

    targets = np.array(full_train.targets)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
    train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

    train_split = Subset(full_train, train_idx)
    val_split = Subset(full_train_eval, val_idx)
    train_unaugmented_split = Subset(full_train_eval, train_idx)

    return {
        "train": train_split,
        "val": val_split,
        "train_unaugmented": train_unaugmented_split,
        "test": test_set
    }

def get_cifar100_unknowns(data_root="./data"):
    _, eval_tf = get_cifar_transforms(use_randaugment=False)
    cifar100_test = datasets.CIFAR100(root=data_root, train=False, download=True, transform=eval_tf)

    fine_to_idx = {name: i for i, name in enumerate(cifar100_test.classes)}
    near_indices_set = {fine_to_idx[name] for name in NEAR_UNKNOWN_CLASSES}
    far_indices_set = {fine_to_idx[name] for name in FAR_UNKNOWN_CLASSES}

    near_samples, far_samples = [], []
    for idx, (img, target) in enumerate(cifar100_test):
        if target in near_indices_set:
            near_samples.append((idx, cifar100_test.classes[target]))
        elif target in far_indices_set:
            far_samples.append((idx, cifar100_test.classes[target]))

    assert len(near_samples) == 800, f"Expected 800 near samples, found {len(near_samples)}"
    assert len(far_samples) == 800, f"Expected 800 far samples, found {len(far_samples)}"

    near_indices = [idx for idx, _ in near_samples]
    far_indices = [idx for idx, _ in far_samples]

    return {
        "near": Subset(cifar100_test, near_indices),
        "far": Subset(cifar100_test, far_indices),
        "near_meta": near_samples,
        "far_meta": far_samples
    }


Writing task4/dataset_protocol.py


In [ ]:
%%writefile task4/models/cifar_resnet.py
import torch
import torch.nn as nn
from torchvision.models.resnet import BasicBlock

class CIFARResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.inplanes = 64

        # CIFAR modification: 3x3 conv1, stride 1, padding 1, no initial maxpool
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion)
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward_features(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        return torch.flatten(out, 1)

    def forward(self, x):
        feat = self.forward_features(x)
        logits = self.fc(feat)
        return logits, feat

    # Sub-network split for PROSER manifold mixup after layer2
    def forward_pre_layer2(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        return out

    def forward_post_layer2(self, h):
        out = self.layer3(h)
        out = self.layer4(out)
        out = self.avgpool(out)
        feat = torch.flatten(out, 1)
        logits = self.fc(feat)
        return logits, feat


Writing task4/models/cifar_resnet.py


## 3. PROSER Implementation: Placeholders & Manifold Mixup (`%%writefile`)

In [ ]:
%%writefile task4/methods/proser.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class PROSERModel(nn.Module):
    def __init__(self, base_model, num_known=10, num_dummies=5):
        super().__init__()
        self.base_model = base_model
        self.num_known = num_known
        self.num_dummies = num_dummies

        # Retain original known fc, attach randomly initialized dummy classifiers
        self.dummy_fc = nn.Linear(512, num_dummies)
        nn.init.kaiming_normal_(self.dummy_fc.weight, mode="fan_out", nonlinearity="relu")
        nn.init.constant_(self.dummy_fc.bias, 0)

    def forward_pre_layer2(self, x):
        return self.base_model.forward_pre_layer2(x)

    def forward_post_layer2(self, h):
        out = self.base_model.layer3(h)
        out = self.base_model.layer4(out)
        out = self.base_model.avgpool(out)
        feat = torch.flatten(out, 1)
        logits_known = self.base_model.fc(feat)
        logits_dummy = self.dummy_fc(feat)
        logits_all = torch.cat([logits_known, logits_dummy], dim=1)
        return logits_all, feat

    def forward(self, x):
        feat = self.base_model.forward_features(x)
        logits_known = self.base_model.fc(feat)
        logits_dummy = self.dummy_fc(feat)
        logits_all = torch.cat([logits_known, logits_dummy], dim=1)
        return logits_all, feat

class PROSERLoss(nn.Module):
    def __init__(self, num_known=10, num_dummies=5, beta=1.0, gamma=0.1):
        super().__init__()
        self.num_known = num_known
        self.num_dummies = num_dummies
        self.beta = beta
        self.gamma = gamma

    def forward(self, logits_cls, labels_cls, logits_data=None):
        # 1. Standard classification on known classes
        loss_ce = F.cross_entropy(logits_cls[:, :self.num_known], labels_cls)

        # 2. Classifier-placeholder loss:
        # Mask ground-truth class, encourage dummy classifiers to dominate remaining logits
        batch_size = logits_cls.size(0)
        masked_logits = logits_cls.clone()
        masked_logits[torch.arange(batch_size), labels_cls] = -1e9
        prob_masked = F.softmax(masked_logits, dim=1)
        dummy_prob_sum = prob_masked[:, self.num_known:].sum(dim=1).clamp(min=1e-8)
        loss_cp = -torch.log(dummy_prob_sum).mean()

        total_loss = loss_ce + self.beta * loss_cp

        # 3. Data-placeholder loss (manifold mixup representations trained to dummy classifiers)
        loss_dp = torch.tensor(0.0, device=logits_cls.device)
        if logits_data is not None and logits_data.size(0) > 0:
            prob_data = F.softmax(logits_data, dim=1)
            dummy_prob_data = prob_data[:, self.num_known:].sum(dim=1).clamp(min=1e-8)
            loss_dp = -torch.log(dummy_prob_data).mean()
            total_loss = total_loss + self.gamma * loss_dp

        return total_loss, {
            "loss_ce": loss_ce.item(),
            "loss_cp": loss_cp.item(),
            "loss_dp": loss_dp.item(),
            "total_loss": total_loss.item()
        }


Writing task4/methods/proser.py


## 4. Novelty Scoring & Evaluation Metrics (`%%writefile`)

In [ ]:
%%writefile task4/evaluation/scores.py
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

def compute_scores(logits_10, features, means_c=None, inv_cov_diag=None):
    """
    Computes all standard unknownness scores on 10 known logits and features.
    Higher value indicates higher novelty / unknownness.
    """
    probs = F.softmax(logits_10, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    u_msp = 1.0 - max_probs

    max_logits, _ = torch.max(logits_10, dim=1)
    u_mls = -max_logits

    u_energy = -torch.logsumexp(logits_10, dim=1)

    scores = {
        "MSP": u_msp.cpu().numpy(),
        "MLS": u_mls.cpu().numpy(),
        "Energy": u_energy.cpu().numpy()
    }

    if means_c is not None and inv_cov_diag is not None:
        # Mahalanobis distance with shared diagonal covariance: min_c (f - mu_c)^T Sigma^{-1} (f - mu_c)
        # features: [N, D], means_c: [10, D], inv_cov_diag: [D]
        feats = features.unsqueeze(1) # [N, 1, D]
        means = means_c.unsqueeze(0)  # [1, 10, D]
        diff = feats - means          # [N, 10, D]
        inv_cov = inv_cov_diag.unsqueeze(0).unsqueeze(0) # [1, 1, D]
        dist_c = torch.sum(diff * inv_cov * diff, dim=2) # [N, 10]
        u_mah, _ = torch.min(dist_c, dim=1)
        scores["Mahalanobis"] = u_mah.cpu().numpy()

    return scores

def compute_proser_placeholder_score(logits_15, num_known=10):
    """
    PROSER placeholder novelty score:
    u(x) = max_dummy(z) - max_known(z)
    """
    known_max, _ = torch.max(logits_15[:, :num_known], dim=1)
    dummy_max, _ = torch.max(logits_15[:, num_known:], dim=1)
    u_proser = dummy_max - known_max
    return u_proser.cpu().numpy()

def compute_osr_metrics(known_scores, unknown_scores, tau_95):
    """
    known_scores: novelty scores on CIFAR-10 test (negative class = 0)
    unknown_scores: novelty scores on CIFAR-100 unknown subset (positive class = 1)
    tau_95: rejection threshold calibrated on 95th percentile of known validation scores
    """
    y_true = np.concatenate([np.zeros(len(known_scores)), np.ones(len(unknown_scores))])
    y_scores = np.concatenate([known_scores, unknown_scores])
    auroc = roc_auc_score(y_true, y_scores) * 100.0

    # Decision: Accept if score <= tau_95, Reject if score > tau_95
    known_accepted = np.mean(known_scores <= tau_95) * 100.0
    unknown_rejected = np.mean(unknown_scores > tau_95) * 100.0
    fpr_95tpr = np.mean(unknown_scores <= tau_95) * 100.0

    return {
        "AUROC": auroc,
        "Known_Acc_Rate": known_accepted,
        "Rejection_Rate": unknown_rejected,
        "FPR@95TPR": fpr_95tpr
    }


Writing task4/evaluation/scores.py


## 5. Training Pipelines: Vanilla, GCSC, and PROSER (`%%writefile`)

In [ ]:
%%writefile task4/train_models.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from task4.dataset_protocol import set_seed, SEED, get_cifar10_splits
from task4.models.cifar_resnet import CIFARResNet18
from task4.methods.proser import PROSERModel, PROSERLoss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate_accuracy(model, loader, device, num_known=10):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _ = model(imgs)
            preds = logits[:, :num_known].argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return (correct / total) * 100.0

def train_standard_classifier(name="vanilla", use_randaugment=False):
    set_seed(SEED)
    splits = get_cifar10_splits("./data", use_randaugment=use_randaugment)
    train_loader = DataLoader(splits["train"], batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(splits["val"], batch_size=128, shuffle=False, num_workers=2)

    model = CIFARResNet18(num_classes=10).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    best_val_acc = -1.0
    best_state = None

    print(f"\n--- Training {name.upper()} (100 Epochs, lr=0.1, Cosine Decay) ---")
    for epoch in range(1, 101):
        model.train()
        total_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)

        scheduler.step()
        val_acc = evaluate_accuracy(model, val_loader, DEVICE)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 20 == 0 or epoch == 100:
            avg_loss = total_loss / len(splits["train"])
            print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}% (Best: {best_val_acc:.2f}%)")

    ckpt_path = f"task4/results/checkpoints/{name}_resnet18.pth"
    torch.save(best_state, ckpt_path)
    print(f"Saved {name} model checkpoint: {ckpt_path}")
    return ckpt_path

def train_proser():
    set_seed(SEED)
    vanilla_ckpt = "task4/results/checkpoints/vanilla_resnet18.pth"
    assert os.path.exists(vanilla_ckpt), "Vanilla checkpoint required before training PROSER."

    base_model = CIFARResNet18(num_classes=10).to(DEVICE)
    base_model.load_state_dict(torch.load(vanilla_ckpt, map_location=DEVICE))
    proser_model = PROSERModel(base_model, num_known=10, num_dummies=5).to(DEVICE)

    splits = get_cifar10_splits("./data", use_randaugment=False)
    train_loader = DataLoader(splits["train"], batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(splits["val"], batch_size=128, shuffle=False, num_workers=2)

    criterion = PROSERLoss(num_known=10, num_dummies=5, beta=1.0, gamma=0.1)
    optimizer = torch.optim.SGD(proser_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_val_acc = -1.0
    best_state = None

    print("\n--- Fine-tuning PROSER (50 Epochs, lr=1e-3, 5 Dummy Classifiers, Manifold Mixup at Layer2) ---")
    for epoch in range(1, 51):
        proser_model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            b = imgs.size(0)
            if b < 4:
                continue
            b_half = b // 2

            # First half: Classifier-placeholder loss
            x_cp, y_cp = imgs[:b_half], labels[:b_half]
            logits_cp, _ = proser_model(x_cp)

            # Second half: Data-placeholder loss via manifold mixup after layer2
            x_dp, y_dp = imgs[b_half:], labels[b_half:]
            n_dp = x_dp.size(0)
            perm = torch.randperm(n_dp, device=DEVICE)
            diff_mask = y_dp != y_dp[perm]

            logits_dp = None
            if diff_mask.sum() > 0:
                h_pre = proser_model.forward_pre_layer2(x_dp)
                h1 = h_pre[diff_mask]
                h2 = h_pre[perm][diff_mask]

                # Beta(2, 2) sampling
                lam = np.random.beta(2.0, 2.0)
                h_mixed = lam * h1 + (1.0 - lam) * h2
                logits_dp, _ = proser_model.forward_post_layer2(h_mixed)

            optimizer.zero_grad()
            loss, _ = criterion(logits_cp, y_cp, logits_dp)
            loss.backward()
            optimizer.step()

        scheduler.step()
        val_acc = evaluate_accuracy(proser_model, val_loader, DEVICE, num_known=10)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(proser_model.state_dict())

        if epoch % 10 == 0 or epoch == 50:
            print(f"PROSER Epoch {epoch:02d} | Val CSA: {val_acc:.2f}% (Best: {best_val_acc:.2f}%)")

    ckpt_path = "task4/results/checkpoints/proser_resnet18.pth"
    torch.save(best_state, ckpt_path)
    print(f"Saved PROSER model checkpoint: {ckpt_path}")
    return ckpt_path

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=["vanilla", "gcsc", "proser"], required=True)
    args = parser.parse_args()

    if args.model == "vanilla":
        train_standard_classifier("vanilla", use_randaugment=False)
    elif args.model == "gcsc":
        train_standard_classifier("gcsc", use_randaugment=True)
    elif args.model == "proser":
        train_proser()


Writing task4/train_models.py


## 6. Execution: Train All Three Models

In [ ]:
# 1. Train Vanilla Closed-Set Baseline (100 epochs)
!python3 task4/train_models.py --model vanilla

100% 170M/170M [38:23<00:00, 74.0kB/s]

--- Training VANILLA (100 Epochs, lr=0.1, Cosine Decay) ---
Epoch 020 | Train Loss: 0.3860 | Val Acc: 77.38% (Best: 85.02%)
Epoch 040 | Train Loss: 0.2688 | Val Acc: 83.34% (Best: 86.94%)
Epoch 060 | Train Loss: 0.1492 | Val Acc: 90.52% (Best: 90.78%)
Epoch 080 | Train Loss: 0.0199 | Val Acc: 94.36% (Best: 94.36%)
Epoch 100 | Train Loss: 0.0025 | Val Acc: 95.50% (Best: 95.58%)
Saved vanilla model checkpoint: task4/results/checkpoints/vanilla_resnet18.pth


In [ ]:
# 2. Train GCSC with RandAugment(2, 9) (100 epochs)
!python3 task4/train_models.py --model gcsc


--- Training GCSC (100 Epochs, lr=0.1, Cosine Decay) ---
Epoch 020 | Train Loss: 0.5309 | Val Acc: 83.40% (Best: 83.96%)
Epoch 040 | Train Loss: 0.4003 | Val Acc: 86.00% (Best: 87.36%)
Epoch 060 | Train Loss: 0.2722 | Val Acc: 90.40% (Best: 90.88%)
Epoch 080 | Train Loss: 0.1098 | Val Acc: 94.10% (Best: 94.38%)
Epoch 100 | Train Loss: 0.0350 | Val Acc: 95.48% (Best: 95.70%)
Saved gcsc model checkpoint: task4/results/checkpoints/gcsc_resnet18.pth


In [ ]:
# 3. Train PROSER (50 epochs fine-tuning with classifier & data placeholders)
!python3 task4/train_models.py --model proser


--- Fine-tuning PROSER (50 Epochs, lr=1e-3, 5 Dummy Classifiers, Manifold Mixup at Layer2) ---
PROSER Epoch 10 | Val CSA: 95.10% (Best: 95.36%)
PROSER Epoch 20 | Val CSA: 95.06% (Best: 95.36%)
PROSER Epoch 30 | Val CSA: 95.28% (Best: 95.36%)
PROSER Epoch 40 | Val CSA: 95.18% (Best: 95.36%)
PROSER Epoch 50 | Val CSA: 95.18% (Best: 95.38%)
Saved PROSER model checkpoint: task4/results/checkpoints/proser_resnet18.pth


In [ ]:
import os
import shutil
from google.colab import drive , files
drive.mount('/content/drive')

Mounted at /content/drive


## 7. Unified Evaluation, Tables & Failure Analysis (`%%writefile`)

In [ ]:
%%writefile task4/evaluation/run_eval.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from task4.dataset_protocol import get_cifar10_splits, get_cifar100_unknowns
from task4.models.cifar_resnet import CIFARResNet18
from task4.methods.proser import PROSERModel
from task4.evaluation.scores import compute_scores, compute_proser_placeholder_score, compute_osr_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def extract_outputs(model, loader, device):
    model.eval()
    logits_list, feats_list, labels_list = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits, feats = model(imgs)
        logits_list.append(logits.cpu())
        feats_list.append(feats.cpu())
        labels_list.append(labels)
    return torch.cat(logits_list, dim=0), torch.cat(feats_list, dim=0), torch.cat(labels_list, dim=0)

def compute_mahalanobis_parameters(features_train, labels_train, num_classes=10):
    D = features_train.size(1)
    means_c = torch.zeros(num_classes, D)
    diffs = []
    for c in range(num_classes):
        mask = labels_train == c
        f_c = features_train[mask]
        mu_c = f_c.mean(dim=0)
        means_c[c] = mu_c
        diffs.append(f_c - mu_c.unsqueeze(0))

    all_diffs = torch.cat(diffs, dim=0) # [N, D]
    shared_var = torch.mean(all_diffs ** 2, dim=0) + 1e-6
    inv_cov_diag = 1.0 / shared_var
    return means_c, inv_cov_diag

def run_full_osr_benchmark():
    cifar10 = get_cifar10_splits("./data", use_randaugment=False)
    cifar100 = get_cifar100_unknowns("./data")

    val_loader = DataLoader(cifar10["val"], batch_size=128, shuffle=False, num_workers=2)
    test_loader = DataLoader(cifar10["test"], batch_size=128, shuffle=False, num_workers=2)
    train_unaugmented_loader = DataLoader(cifar10["train_unaugmented"], batch_size=128, shuffle=False, num_workers=2)

    near_loader = DataLoader(cifar100["near"], batch_size=128, shuffle=False, num_workers=2)
    far_loader = DataLoader(cifar100["far"], batch_size=128, shuffle=False, num_workers=2)

    # -----------------------------------------------------------------------------------
    # Step 1 & 2: Compare MSP, MLS, Energy, Mahalanobis on Frozen Vanilla Model
    # -----------------------------------------------------------------------------------
    vanilla = CIFARResNet18(num_classes=10).to(DEVICE)
    vanilla.load_state_dict(torch.load("task4/results/checkpoints/vanilla_resnet18.pth", map_location=DEVICE))

    # Extract features for Mahalanobis calibration
    tr_logits, tr_feats, tr_labels = extract_outputs(vanilla, train_unaugmented_loader, DEVICE)
    means_c, inv_cov_diag = compute_mahalanobis_parameters(tr_feats, tr_labels)

    val_logits, val_feats, _ = extract_outputs(vanilla, val_loader, DEVICE)
    test_logits, test_feats, test_labels = extract_outputs(vanilla, test_loader, DEVICE)
    near_logits, near_feats, _ = extract_outputs(vanilla, near_loader, DEVICE)
    far_logits, far_feats, _ = extract_outputs(vanilla, far_loader, DEVICE)

    val_scores_vanilla = compute_scores(val_logits, val_feats, means_c, inv_cov_diag)
    test_scores_vanilla = compute_scores(test_logits, test_feats, means_c, inv_cov_diag)
    near_scores_vanilla = compute_scores(near_logits, near_feats, means_c, inv_cov_diag)
    far_scores_vanilla = compute_scores(far_logits, far_feats, means_c, inv_cov_diag)

    table1_rows = []
    score_names = ["MSP", "MLS", "Energy", "Mahalanobis"]
    for sc in score_names:
        # Calibrate rejection threshold at 95th percentile of validation unknownness
        tau_95 = float(np.percentile(val_scores_vanilla[sc], 95.0))
        all_unknown = np.concatenate([near_scores_vanilla[sc], far_scores_vanilla[sc]])

        m_near = compute_osr_metrics(test_scores_vanilla[sc], near_scores_vanilla[sc], tau_95)
        m_far = compute_osr_metrics(test_scores_vanilla[sc], far_scores_vanilla[sc], tau_95)
        m_all = compute_osr_metrics(test_scores_vanilla[sc], all_unknown, tau_95)

        table1_rows.append({
            "Score": sc,
            "Near AUROC": round(m_near["AUROC"], 2),
            "Far AUROC": round(m_far["AUROC"], 2),
            "All AUROC": round(m_all["AUROC"], 2),
            "Test Acc Rate (%)": round(m_all["Known_Acc_Rate"], 2),
            "Near FPR@95TPR": round(m_near["FPR@95TPR"], 2),
            "Far FPR@95TPR": round(m_far["FPR@95TPR"], 2),
            "All Rejection (%)": round(m_all["Rejection_Rate"], 2)
        })

    df_table1 = pd.DataFrame(table1_rows)
    df_table1.to_csv("task4/results/table1_vanilla_scores_comparison.csv", index=False)
    print("\n" + "="*95)
    print("TABLE 1: POST-HOC NOVELTY SCORES ON FROZEN VANILLA MODEL")
    print("="*95)
    print(df_table1.to_string(index=False))

    # -----------------------------------------------------------------------------------
    # Step 3 & 4: Model Comparison (Vanilla, GCSC, PROSER)
    # -----------------------------------------------------------------------------------
    # Load GCSC
    gcsc = CIFARResNet18(num_classes=10).to(DEVICE)
    gcsc.load_state_dict(torch.load("task4/results/checkpoints/gcsc_resnet18.pth", map_location=DEVICE))
    val_logits_g, _, _ = extract_outputs(gcsc, val_loader, DEVICE)
    test_logits_g, _, test_labels_g = extract_outputs(gcsc, test_loader, DEVICE)
    near_logits_g, _, _ = extract_outputs(gcsc, near_loader, DEVICE)
    far_logits_g, _, _ = extract_outputs(gcsc, far_loader, DEVICE)

    # Load PROSER
    base_p = CIFARResNet18(num_classes=10).to(DEVICE)
    proser = PROSERModel(base_p, num_known=10, num_dummies=5).to(DEVICE)
    proser.load_state_dict(torch.load("task4/results/checkpoints/proser_resnet18.pth", map_location=DEVICE))
    val_logits_p, _, _ = extract_outputs(proser, val_loader, DEVICE)
    test_logits_p, _, test_labels_p = extract_outputs(proser, test_loader, DEVICE)
    near_logits_p, _, _ = extract_outputs(proser, near_loader, DEVICE)
    far_logits_p, _, _ = extract_outputs(proser, far_loader, DEVICE)

    models_eval = [
        ("Vanilla", test_logits, test_labels, val_scores_vanilla["MLS"], test_scores_vanilla["MLS"], near_scores_vanilla["MLS"], far_scores_vanilla["MLS"], "MLS"),
        ("GCSC", test_logits_g, test_labels_g, compute_scores(val_logits_g, None)["MLS"], compute_scores(test_logits_g, None)["MLS"], compute_scores(near_logits_g, None)["MLS"], compute_scores(far_logits_g, None)["MLS"], "MLS"),
        ("PROSER (MLS)", test_logits_p[:, :10], test_labels_p, compute_scores(val_logits_p[:, :10], None)["MLS"], compute_scores(test_logits_p[:, :10], None)["MLS"], compute_scores(near_logits_p[:, :10], None)["MLS"], compute_scores(far_logits_p[:, :10], None)["MLS"], "MLS"),
        ("PROSER (Placeholder)", test_logits_p[:, :10], test_labels_p, compute_proser_placeholder_score(val_logits_p), compute_proser_placeholder_score(test_logits_p), compute_proser_placeholder_score(near_logits_p), compute_proser_placeholder_score(far_logits_p), "Placeholder")
    ]

    table2_rows = []
    for m_name, t_logits_10, t_labels, v_sc, t_sc, n_sc, f_sc, sc_type in models_eval:
        csa = (t_logits_10.argmax(dim=1) == t_labels).float().mean().item() * 100.0
        tau_95 = float(np.percentile(v_sc, 95.0))
        all_sc = np.concatenate([n_sc, f_sc])

        m_near = compute_osr_metrics(t_sc, n_sc, tau_95)
        m_far = compute_osr_metrics(t_sc, f_sc, tau_95)
        m_all = compute_osr_metrics(t_sc, all_sc, tau_95)

        table2_rows.append({
            "Model": m_name,
            "Score": sc_type,
            "CSA (%)": round(csa, 2),
            "Near AUROC": round(m_near["AUROC"], 2),
            "Far AUROC": round(m_far["AUROC"], 2),
            "All AUROC": round(m_all["AUROC"], 2),
            "Near FPR@95TPR": round(m_near["FPR@95TPR"], 2),
            "Far FPR@95TPR": round(m_far["FPR@95TPR"], 2),
            "All Rejection (%)": round(m_all["Rejection_Rate"], 2)
        })

    df_table2 = pd.DataFrame(table2_rows)
    df_table2.to_csv("task4/results/table2_models_comparison.csv", index=False)
    print("\n" + "="*95)
    print("TABLE 2: CLOSED-SET ACCURACY VS OPEN-SET REJECTION ACROSS MODELS")
    print("="*95)
    print(df_table2.to_string(index=False))

    # -----------------------------------------------------------------------------------
    # Multi-panel ROC Plot (MSP, MLS, Mahalanobis on Vanilla)
    # -----------------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_scores = [("MSP", test_scores_vanilla["MSP"], near_scores_vanilla["MSP"], far_scores_vanilla["MSP"]),
                   ("MLS", test_scores_vanilla["MLS"], near_scores_vanilla["MLS"], far_scores_vanilla["MLS"]),
                   ("Mahalanobis", test_scores_vanilla["Mahalanobis"], near_scores_vanilla["Mahalanobis"], far_scores_vanilla["Mahalanobis"])]

    for idx, (title, k_sc, n_sc, f_sc) in enumerate(plot_scores):
        ax = axes[idx]
        # Near ROC
        y_n = np.concatenate([np.zeros(len(k_sc)), np.ones(len(n_sc))])
        fpr_n, tpr_n, _ = roc_curve(y_n, np.concatenate([k_sc, n_sc]))
        auc_n = compute_osr_metrics(k_sc, n_sc, 0.0)["AUROC"]
        ax.plot(fpr_n, tpr_n, label=f"Near (AUC={auc_n:.1f}%)", color="tab:orange", lw=2)

        # Far ROC
        y_f = np.concatenate([np.zeros(len(k_sc)), np.ones(len(f_sc))])
        fpr_f, tpr_f, _ = roc_curve(y_f, np.concatenate([k_sc, f_sc]))
        auc_f = compute_osr_metrics(k_sc, f_sc, 0.0)["AUROC"]
        ax.plot(fpr_f, tpr_f, label=f"Far (AUC={auc_f:.1f}%)", color="tab:blue", lw=2)

        ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
        ax.set_title(f"{title} ROC Curves", fontsize=13)
        ax.set_xlabel("False Positive Rate (FPR)")
        ax.set_ylabel("True Positive Rate (TPR)")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="lower right")

    plt.tight_layout()
    roc_fig_path = "task4/results/plots/roc_comparison.png"
    plt.savefig(roc_fig_path, dpi=300)
    print(f"\nSaved ROC comparison plot to: {roc_fig_path}")
    plt.show()

    # -----------------------------------------------------------------------------------
    # Failure Analysis: Inspect 3 Near and 3 Far Incorrectly Accepted Unknowns
    # -----------------------------------------------------------------------------------
    c10_classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
    tau_mls = float(np.percentile(val_scores_vanilla["MLS"], 95.0))

    print("\n" + "="*95)
    print(f"FAILURE ANALYSIS: UNKNOWN EXAMPLES INCORRECTLY ACCEPTED UNDER VANILLA MLS (tau={tau_mls:.4f})")
    print("="*95)

    def find_failures(scores, logits, meta, group_name):
        # Incorrect acceptance: u(x) <= tau_mls
        accepted_mask = scores <= tau_mls
        indices = np.where(accepted_mask)[0]
        # Sort by lowest score (highest confidence / most catastrophic acceptance)
        sorted_order = indices[np.argsort(scores[indices])]

        print(f"\nTop Incorrectly Accepted [{group_name.upper()} UNKNOWNS] (Accepted {len(indices)}/{len(scores)}):")
        for rank, idx in enumerate(sorted_order[:3], 1):
            true_cls = meta[idx][1]
            pred_c10 = c10_classes[logits[idx].argmax().item()]
            sc = scores[idx]
            print(f"  {rank}. CIFAR-100 Class: '{true_cls}' -> Confidently Predicted as CIFAR-10: '{pred_c10}' | MLS Score: {sc:.4f} (Threshold: {tau_mls:.4f})")

    find_failures(near_scores_vanilla["MLS"], near_logits, cifar100["near_meta"], "Near")
    find_failures(far_scores_vanilla["MLS"], far_logits, cifar100["far_meta"], "Far")

if __name__ == "__main__":
    run_full_osr_benchmark()


Writing task4/evaluation/run_eval.py


## 8. Execute Full Evaluation, Generate Tables, and Inspect Failures

In [ ]:
!python3 task4/evaluation/run_eval.py

100% 169M/169M [35:46<00:00, 78.7kB/s]

TABLE 1: POST-HOC NOVELTY SCORES ON FROZEN VANILLA MODEL
      Score  Near AUROC  Far AUROC  All AUROC  Test Acc Rate (%)  Near FPR@95TPR  Far FPR@95TPR  All Rejection (%)
        MSP       82.50      89.30      85.90              94.09           67.88          52.00              40.06
        MLS       81.34      89.23      85.28              94.44           65.62          47.00              43.69
     Energy       81.35      89.26      85.31              94.61           66.88          47.00              43.06
Mahalanobis       79.87      91.55      85.71              94.29           70.12          44.62              42.62

TABLE 2: CLOSED-SET ACCURACY VS OPEN-SET REJECTION ACROSS MODELS
               Model       Score  CSA (%)  Near AUROC  Far AUROC  All AUROC  Near FPR@95TPR  Far FPR@95TPR  All Rejection (%)
             Vanilla         MLS    94.41       81.34      89.23      85.28           65.62          47.00              43.69
          

In [ ]:
# 2. Mirror full task4 directory into Drive
target_drive_task4 = "/content/drive/MyDrive/ATML_PA1/Task4"
os.makedirs(target_drive_task4, exist_ok=True)
shutil.copytree("task4", target_drive_task4, dirs_exist_ok=True)
print(f"Direct folder sync complete: 'task4' -> '{target_drive_task4}'")

# 3. Create zip archives for checkpoints + code and lightweight results
all_zip_path = os.path.join(target_drive_task4, "task4_complete_backup.zip")
results_only_zip = os.path.join(target_drive_task4, "task4_results_tables_plots.zip")

shutil.make_archive(all_zip_path.replace(".zip", ""), "zip", root_dir=".", base_dir="task4")
shutil.make_archive(results_only_zip.replace(".zip", ""), "zip", root_dir="task4", base_dir="results")
print(f"Archives saved to Drive:\n - {all_zip_path}\n - {results_only_zip}")

# 4. Trigger browser download of all evaluation tables and plots
local_results_zip = "task4_results_tables_plots.zip"
shutil.make_archive("task4_results_tables_plots", "zip", root_dir="task4", base_dir="results")
print("Downloading results archive to your local machine...")
files.download(local_results_zip)

Direct folder sync complete: 'task4' -> '/content/drive/MyDrive/ATML_PA1/Task4'
Archives saved to Drive:
 - /content/drive/MyDrive/ATML_PA1/Task4/task4_complete_backup.zip
 - /content/drive/MyDrive/ATML_PA1/Task4/task4_results_tables_plots.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>